In [21]:
import msgpack, msgpack_numpy, numpy as np, requests
from droid_plus.policies.image_tools import resize_with_pad

POLICY_URL = "http://127.0.0.1:8000"     # your SSH tunnel endpoint
PROMPT     = "lift the white color object"

print("health:", requests.get(f"{POLICY_URL}/health", timeout=10).json())

def infer(left, wrist, q, grip, prompt=PROMPT):
    req = {
        "images": {
            "left":  resize_with_pad(np.asarray(left,  np.uint8), 224, 224),
            "wrist": resize_with_pad(np.asarray(wrist, np.uint8), 224, 224),
        },
        "state":  np.concatenate([np.asarray(q, np.float32).reshape(-1), np.float32([grip])]),
        "prompt": prompt,
    }
    blob = msgpack.packb(req, default=msgpack_numpy.encode, use_bin_type=True)
    r = requests.post(f"{POLICY_URL}/infer", data=blob,
                      headers={"Content-Type": "application/msgpack"}, timeout=20)
    r.raise_for_status()
    out = msgpack.unpackb(r.content, object_hook=msgpack_numpy.decode, raw=False)
    return np.asarray(out["actions"], dtype=float)


health: {'ok': True, 'checkpoint': '/home/developer/lerobot_ft/outputs/lift_white_color_object_teleop_20260909_190957/checkpoints/last/pretrained_model/', 'device': 'cuda', 'action_dim': 8}


In [62]:
import concurrent.futures as cf
import time
executor = cf.ThreadPoolExecutor(max_workers=1)
pending = None
PREFETCH_AT = 5
EXECUTE_STEPS = 10

In [58]:
Q_MIN = np.array([-2.7437,-1.7837,-2.9007,-3.0421,-2.8065, 0.5445,-3.0159])
Q_MAX = np.array([ 2.7437, 1.7837, 2.9007,-0.1518, 2.8065, 4.5169, 3.0159])
MARGIN = 0.12

In [101]:
from droid_plus.robot import DroidPlus
droid = DroidPlus()
g = droid.gripper

gripper initialized None


In [56]:
def get_obs():
    left  = droid.get_left_image(jpeg_quality=90)
    wrist = droid.get_wrist_image(jpeg_quality=90)
    q     = np.asarray(droid.get_current_joint_state()["positions"], float)
    try:    grip = float(g.gripper_position_frac())
    except Exception: grip = 0.0
    return left, wrist, q, grip

In [61]:
left, wrist, q, grip = get_obs()

In [ ]:
droid.go_home()

{'positions': [0.0, 0.0, 0.0, -1.571, 0.0, 1.571, 0.0],
 'velocities': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 'seq': None,
 'accepted_timestamp_s': 1789162525.843777,
 'age_s': 0.0,
 'stop_latched': False}

In [48]:
TCP_OFFSET_Z    = 0.16 
Z_FLOOR = -0.034 #table actual floor is  -0.042
_FR3_DH = [
    (0.0,     0.333,  0.0),
    (0.0,     0.0,   -np.pi / 2),
    (0.0,     0.316,  np.pi / 2),
    (0.0825,  0.0,    np.pi / 2),
    (-0.0825, 0.384, -np.pi / 2),
    (0.0,     0.0,    np.pi / 2),
    (0.088,   0.0,    np.pi / 2),
    (0.0,     0.107,  0.0),
]

def _fr3_flange_T(q):
    """4x4 base->flange (panda_link8) transform for arm joints q (7,)."""
    q = np.asarray(q, float).reshape(-1)[:7]
    T = np.eye(4)
    for (a, d, alpha), th in zip(_FR3_DH, list(q) + [0.0]):
        ca, sa, ct, st = np.cos(alpha), np.sin(alpha), np.cos(th), np.sin(th)
        T = T @ np.array([
            [ct,    -st,    0.0,  a],
            [st*ca,  ct*ca, -sa, -sa*d],
            [st*sa,  ct*sa,  ca,  ca*d],
            [0.0,    0.0,    0.0,  1.0],
        ])
    return T

def gripper_z(q):
    """Base-frame height (m) of the gripper tip for arm joint vector q (7,)."""
    T = _fr3_flange_T(q)
    return float(T[2, 3] + T[2, 2] * TCP_OFFSET_Z)   # flange origin + Z_flange * offset


In [49]:
gripper_z(q)

-0.03211137284433546

In [72]:
chunk = infer(*get_obs())

In [107]:
droid.go_home()

{'positions': [0.0, 0.0, 0.0, -1.571, 0.0, 1.571, 0.0],
 'velocities': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 'seq': None,
 'accepted_timestamp_s': 1789165162.8846843,
 'age_s': 0.0,
 'stop_latched': False}

In [105]:
g.open()

{'ok': True, 'position': 0, 'object_detected': False, 'accepted': False}

In [108]:
CONTROL_HZ = 15.0
DT         = 1.0 / CONTROL_HZ
STEPS      = 15             # execute only the first 10 actions of the chunk
MAX_STEP   = 0.05           # rad/tick clamp toward each target (absorbs the q->action[0] gap)
STEP_ABORT = 0.6          # abort if any single step target is this far from current q
MARGIN     = 0.12
GRIP_SPEED = 120

for i in range(5):
    print(i)
    chunk = infer(*get_obs())        # (>=10, 8): cols 0-6 = joint target, col 7 = gripper
    tgt   = np.clip(chunk[:STEPS, :7], Q_MIN + MARGIN, Q_MAX - MARGIN)
    z_cand = gripper_z(tgt)
    if z_cand < Z_FLOOR:
                raise RuntimeError(f"TABLE SAFETY: chunk {i} step {k} would put the gripper tip at "
                                    f"z={z_cand:+.3f} m < floor {Z_FLOOR:+.3f} m — stopping")
    gplan = chunk[:STEPS, 7]
    q_now = np.asarray(droid.get_current_joint_state()["positions"], float)
    step_max = np.abs(np.diff(np.vstack([q_now, tgt]), axis=0)).max()
    if step_max > STEP_ABORT:
        raise RuntimeError(f"chunk step {step_max:.3f} rad > STEP_ABORT — refusing to execute")

    sent = q.astype(float).copy()
    seq  = 0
    t    = time.time()

    for k in range(STEPS):
        sent = np.clip(sent + np.clip(tgt[k] - sent, -MAX_STEP, MAX_STEP), Q_MIN, Q_MAX)
        droid.set_target_joint_state(sent, velocities=[0.0] * 7, seq=seq)
        seq += 1

        if gplan[k] > 0.5:
            g.close_async(speed=GRIP_SPEED)
        else:
            g.open_async(speed=GRIP_SPEED)

        t += DT
        time.sleep(max(0.0, t - time.time()))

    time.sleep(0.15)              
    droid.stop()
    print("final q:", np.round(droid.get_current_joint_state()["positions"], 3))


0
final q: [-0.065 -0.068  0.102 -2.175 -0.044  2.068  0.696]
1
final q: [-0.106  0.195  0.112 -2.24  -0.073  2.256  0.639]
2
final q: [-0.154  0.319  0.118 -2.19  -0.098  2.203  0.598]
3
final q: [-0.171  0.409  0.13  -2.183 -0.111  2.229  0.59 ]
4
final q: [-0.169  0.485  0.142 -2.132 -0.152  2.204  0.585]
